In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date, timedelta

# 1. Create 50 e-commerce records
customers = [
    "Rahul", "Priya", "Arjun", "Sneha", "Kiran",
    "Ananya", "Vijay", "Divya", "Rohit", "Pooja"
]

products = [
    ("Laptop", "Electronics", 55000.0),
    ("Mobile", "Electronics", 25000.0),
    ("Headphones", "Electronics", 2000.0),
    ("T-Shirt", "Fashion", 799.0),
    ("Shoes", "Fashion", 2499.0),
    ("Watch", "Accessories", 3500.0),
    ("Backpack", "Accessories", 1500.0),
    ("Chair", "Home", 4500.0),
    ("Mixer", "Home", 2800.0),
    ("Book", "Books", 499.0)
]

cities = [
    "Bangalore", "Hyderabad", "Chennai",
    "Mumbai", "Pune", "Delhi"
]

payments = ["UPI", "Credit Card", "Debit Card", "Cash"]
statuses = ["Delivered", "Shipped", "Pending", "Cancelled", "Returned"]

data = []

for i in range(1, 51):
    product = products[(i - 1) % len(products)]

    data.append((
        i,                                      # order_id
        101 + ((i - 1) % 20),                  # customer_id
        customers[(i - 1) % len(customers)],
        product[0],                            # product_name
        product[1],                            # category
        1 + (i % 4),                           # quantity
        product[2],                            # unit_price
        date(2026, 1, 1) + timedelta(days=(i - 1)),
        cities[(i - 1) % len(cities)],
        payments[(i - 1) % len(payments)],
        statuses[(i - 1) % len(statuses)]
    ))

schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("order_date", DateType(), True),
    StructField("city", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("order_status", StringType(), True)
])

customer_orders_df = spark.createDataFrame(data, schema)

# Add a calculated sales amount
customer_orders_df = customer_orders_df.withColumn(
    "total_amount",
    F.round(F.col("quantity") * F.col("unit_price"), 2)
)

display(customer_orders_df)

# Verify the number of records
print("Total rows:", customer_orders_df.count())


In [0]:
%sql

CREATE VOLUME IF NOT EXISTS workspace.default.ecommerce_volume;

In [0]:

# Volume location
volume_path = "/Volumes/workspace/default/ecommerce_volume"

# Save the DataFrame in Delta format
customer_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(volume_path)

print("Delta data saved successfully:", volume_path)


In [0]:

display(dbutils.fs.ls(
    "/Volumes/workspace/default/ecommerce_volume/"
))


In [0]:

customer_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.ecommerce_orders")

print("Delta table created: workspace.default.ecommerce_orders")


In [0]:
%sql
select * from ecommerce_orders

In [0]:
df = spark.table("ecommerce_orders")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("customer_id") \
    .saveAsTable("ecommerce_orders")

In [0]:
%sql
show partitions ecommerce_orders

- Z-Ordering: A technique in Databricks that organizes related data values closer together in the same files, so queries can find data faster by skipping unnecessary files.

- Liquid Clustering: A technique in Databricks that organizes data based on selected columns and can incrementally maintain that organization as data grows, making queries faster and data layout easier to manage.

In [0]:
df = spark.read.json("/Workspace/Users/naveenbommisetty6@gmail.com/DATA ENGINEER/Data_eng/depart.json")

df.show(truncate=False)
df.printSchema()

In [0]:
from pyspark.sql.functions import explode

exploded_df = df.select(
    "id",
    "name",
    explode("skills").alias("skill")
)

exploded_df.show(truncate=False)

### Read mode

PERMISSIVE

Default mode. Reads valid records and puts malformed records into a corrupt-record column if configured; otherwise, malformed fields may become null.

DROPMALFORMED

Drops records that Spark identifies as malformed.

FAILFAST

Throws an exception as soon as a malformed record is encountered.

In [0]:
df = spark.read \
    .option("mode", "PERMISSIVE") \
    .json("/Workspace/Users/naveenbommisetty6@gmail.com/DATA ENGINEER/Data_eng/employees.json")

df.show(truncate=False)

In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType,
    StringType, ArrayType
)

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("skills", ArrayType(StringType()), True),
    StructField("_corrupt_record", StringType(), True)
])

df = spark.read \
    .schema(schema) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json("/Workspace/Users/naveenbommisetty6@gmail.com/DATA ENGINEER/Data_eng/employees.json")

df.show(truncate=False)

In [0]:
df_drop = spark.read \
    .schema(schema) \
    .option("mode", "DROPMALFORMED") \
    .json("/Workspace/Users/naveenbommisetty6@gmail.com/DATA ENGINEER/Data_eng/employees.json")

df_drop.show(truncate=False)

In [0]:
df_fail = spark.read \
    .schema(schema) \
    .option("mode", "FAILFAST") \
    .json("/Workspace/Users/naveenbommisetty6@gmail.com/DATA ENGINEER/Data_eng/employees.json")

df_fail.show(truncate=False)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS default.employees_copy_into (
    id INT,
    name STRING,
    salary INT
)
USING DELTA;

In [0]:
%sql
COPY INTO default.employees_copy_into
FROM (SELECT CAST(id AS INT) AS id, name, CAST(salary AS INT) AS salary FROM '/Volumes/workspace/default/student_volume/raw_files/')
FILEFORMAT = JSON;

In [0]:
%sql
SELECT *
FROM default.employees_copy_into
ORDER BY id;

In [0]:
%sql
select * from bronze_orders

In [0]:
%sql
select * from silver_orders